# 06 · Final pipeline

Two parts, kept separate so that their effects can be told apart:

- **A. Data and inference strategies** with a *fixed* trained model: NMS threshold, test-time augmentation, sliced inference, ensembling (WBF).
- **B. Model / training-data changes** with a *fixed* inference setting: more training frames, full augmentation, longer training.

Finally the chosen configuration is retrained on **all** labelled clips and used for the test submission.

## 1. Setup

In [ ]:
import os, sys, json, shutil
from pathlib import Path

REPO = Path("/kaggle/working/repo") if Path("/kaggle").exists() else Path("..").resolve()
if Path("/kaggle").exists() and not REPO.exists():
    !git clone https://github.com/<user>/<repo>.git {REPO}
    !pip install -q faster-coco-eval ultralytics==8.3.*
sys.path.insert(0, str(REPO / "src"))

import numpy as np, pandas as pd, matplotlib.pyplot as plt, matplotlib.patches as patches, cv2
import vdet, parallel, ultra, infer
from vdet import CLASSES

vdet.seed_everything()
ROOT = vdet.find_data_root()
OUT = Path("/kaggle/working") if Path("/kaggle").exists() else REPO / "outputs"
RUNS = OUT / "runs"; RUNS.mkdir(parents=True, exist_ok=True)
RESULT_DIRS = [RUNS, REPO / "results"]   # own runs + runs committed by the teammate
P = vdet.PROTOCOL

# restore finished runs from earlier versions of this notebook (attached as Kaggle input)
for src in (Path("/kaggle/input").glob("*/runs/*") if Path("/kaggle/input").exists() else []):
    if (src / "config.json").exists() and not (RUNS / src.name).exists():
        shutil.copytree(src, RUNS / src.name); print("restored", src.name)

In [ ]:
imgs, anns = vdet.load_all(ROOT)
folds = pd.read_csv(OUT / "folds.csv") if (OUT / "folds.csv").exists() else vdet.make_folds(imgs, P["n_folds"])
folds.to_csv(OUT / "folds.csv", index=False)   # the parallel runner reads the split from this file
train_frames, val_frames = vdet.protocol_frames(imgs, folds, P)
train_gt = anns[anns.key.isin(train_frames.key) & (anns.iscrowd == 0)]
print(json.dumps(P))
print(f"train: {len(train_frames)} frames | val: {len(val_frames)} frames / {val_frames.clip_uid.nunique()} clips")

COLS = ["AP", "AP50", "AP75", "AP_small", "AP_medium", "AP_large", "AR_max", "ms_per_img", "epochs"]
def show(res, cols=COLS):
    cols = [c for c in cols if c in res]
    num = [c for c in cols if c not in ("ms_per_img", "epochs")]
    return res[cols].style.format("{:.4f}", subset=num).format("{:.1f}", subset=[c for c in ["ms_per_img"] if c in cols])

In [ ]:
# All model results so far (own runs + teammate's committed results/)
allres = vdet.collect_results(RESULT_DIRS)
if len(allres):
    allres["family"] = allres.index.str.split("_").str[0]
    best = allres.loc[allres.groupby("family").AP.idxmax()].sort_values("AP", ascending=False)
    display(show(best, COLS + ["family"]))
    best[["AP", "AP50", "AP_small"]].plot.bar(figsize=(10, 4), rot=0, title="best run per detector family")
    plt.tight_layout(); plt.savefig(OUT / "summary_all_models.png", dpi=120); plt.show()

## 2. Part A — inference strategies on a fixed model

In [ ]:
# The best single model. Its run folder must be available: own RUNS, or attach the teammate's
# Kaggle notebook output as input (the setup cell copies */runs/* automatically).
BASE_RUN = "yolo_y1_p2"
cfg = json.loads((RUNS / BASE_RUN / "config.json").read_text()); IMGSZ = cfg["imgsz"]
model = ultra.load(RUNS / BASE_RUN)
A = {}
def evalA(name, preds):
    A[name] = vdet.coco_evaluate(val_frames, anns, preds, per_class=False); print(name, {k: round(v, 4) for k, v in A[name].items()})
    return preds

In [ ]:
# A1: NMS IoU threshold -- dense scenes have true neighbours that overlap
for iou in [0.5, 0.6, 0.7]:
    evalA(f"A1 nms_iou={iou}", ultra.predict_frames(model, val_frames, IMGSZ, iou=iou))

In [ ]:
NMS_IOU = 0.6   # set from A1
# A2: test-time augmentation (Ultralytics: flips + 2 extra scales, merged before NMS)
evalA("A2 TTA", ultra.predict_frames(model, val_frames, IMGSZ, iou=NMS_IOU, augment=True))

In [ ]:
# A3: sliced inference -- crops are upscaled to IMGSZ by the model, i.e. a zoom-in on tiny objects
fn = lambda ims: ultra.predict_images(model, ims, IMGSZ, iou=NMS_IOU)
for tile in [640, 960]:
    evalA(f"A3 sliced tile={tile}", infer.sliced_predict(fn, val_frames, tile=tile, overlap=0.2, full=True, merge_iou=NMS_IOU))

In [ ]:
# A4: WBF ensemble of the best run of each family (saved val predictions -> no GPU needed)
pred_files = {}
for run in (best.index if len(allres) else []):
    f = next((Path(d) / run / "val_preds.csv.gz" for d in RESULT_DIRS if (Path(d) / run / "val_preds.csv.gz").exists()), None)
    if f is not None: pred_files[run] = f
print("models available for the ensemble:", list(pred_files))
val_preds = {r: pd.read_csv(f) for r, f in pred_files.items()}
order = sorted(val_preds, key=lambda r: -best.loc[r, "AP"])
for k in range(2, len(order) + 1):
    runs = order[:k]
    evalA(f"A4 WBF {'+'.join(runs)}", infer.wbf([val_preds[r] for r in runs], weights=[best.loc[r, "AP"] for r in runs], iou_thr=0.55))

In [ ]:
tabA = pd.DataFrame(A).T[["AP", "AP50", "AP_small", "AP_medium"]]
display(tabA.style.format("{:.4f}").highlight_max(color="#cde"))

## 3. Part B — training data / model changes with fixed inference

In [ ]:
# B1: denser frame sampling (step 5 instead of 10); B2: full Ultralytics augmentation (mosaic, scale, HSV);
# B3: longer training. All on the BASE_RUN architecture, evaluated with plain inference.
data5 = vdet.build_yolo_dataset(imgs, anns, folds, val_fold=P["val_fold"], out_dir=OUT / "yolo_fold0_step5",
                                step=5, val_step=P["val_step"])
data10 = OUT / "yolo_fold0" / "data.yaml"
if not data10.exists():
    data10 = vdet.build_yolo_dataset(imgs, anns, folds, val_fold=P["val_fold"], out_dir=OUT / "yolo_fold0",
                                     step=P["step"], val_step=P["val_step"])
FULL_AUG = dict(mosaic=1.0, scale=0.5, translate=0.1, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, close_mosaic=3)
B = {"final_b1_step5": (data5, {**cfg}),
     "final_b2_fullaug": (data10, {**cfg, "aug": FULL_AUG}),
     "final_b3_24ep": (data10, {**cfg, "epochs": 2 * cfg["epochs"]})}
RUN_B = ["final_b1_step5", "final_b2_fullaug", "final_b3_24ep"]
# each job carries its own dataset; with 2 GPUs b1 + b2 run together, then b3
parallel.run_parallel([(n, B[n][1], B[n][0]) for n in RUN_B], RUNS, folds_csv=OUT / "folds.csv")
display(show(pd.concat([vdet.collect_results(RESULT_DIRS, BASE_RUN), vdet.collect_results(RESULT_DIRS, "final_b")])))

## 4. Final model: chosen settings, trained on all labelled clips

In [ ]:
FINAL_CFG = {**cfg}            # add the winning Part B changes here, e.g. {"aug": FULL_AUG}
FINAL_STEP = 5                 # from Part B
data_all = vdet.build_yolo_dataset(imgs, anns, folds, val_fold=P["val_fold"], out_dir=OUT / "yolo_all",
                                   step=FINAL_STEP, val_step=P["val_step"], train_all=True)
# val metrics of this run are optimistic (holdout is in training); they are only a sanity check
parallel.run_parallel([("final_all_clips", FINAL_CFG)], RUNS, folds_csv=OUT / "folds.csv", data_yaml=data_all)

In [ ]:
# Test predictions with the inference strategy chosen in Part A
final = ultra.load(RUNS / "final_all_clips")
test = pd.read_csv(ROOT / "test_images.csv")
test_frames = test.assign(key=test.image_id, img_path=[str(ROOT / f) for f in test.file_name])
STRATEGY = "plain"   # "plain", "tta" or "sliced", from Part A
if STRATEGY == "sliced":
    fn = lambda ims: ultra.predict_images(final, ims, FINAL_CFG["imgsz"], iou=NMS_IOU)
    tp = infer.sliced_predict(fn, test_frames, tile=640, overlap=0.2, merge_iou=NMS_IOU)
else:
    tp = ultra.predict_frames(final, test_frames, FINAL_CFG["imgsz"], iou=NMS_IOU, augment=STRATEGY == "tta")
tp = tp.rename(columns={"key": "image_id"})
tp.to_csv(OUT / "test_preds_final.csv.gz", index=False)
vdet.write_submission(tp, ROOT / "test_images.csv", OUT / "submission_final.csv")

In [ ]:
# Optional: WBF of the final model with other models' test predictions (files written by notebooks 02-05;
# attach those notebook outputs as inputs). Use only if A4 showed a gain on the holdout.
others = sorted(Path("/kaggle/input").glob("*/test_preds_*.csv.gz")) if Path("/kaggle/input").exists() else []
print([o.name for o in others])
if others:
    parts = [tp] + [pd.read_csv(o) for o in others]
    fused = infer.wbf([p.rename(columns={"image_id": "key"}) for p in parts], iou_thr=0.55).rename(columns={"key": "image_id"})
    vdet.write_submission(fused, ROOT / "test_images.csv", OUT / "submission_final_wbf.csv")

In [ ]:
# Results for the repo: unzip results.zip from the notebook Output into <repo>/results/ and commit.
vdet.export_results(RUNS, OUT / "results", preds_for=("final_all_clips",))